# 04. Models (Python)
**Case study:** Partisanship & economic perception · **Layer:** marts, Python side (regressions and resampling SQL cannot do)

The plan's pipeline rule: *SQL for everything SQL can do, Python only for the steps it can't.* Notebook 03 built every
descriptive mart. This notebook adds the estimates that need a model or a resampling loop, and writes each one back to the
schema as a table, so the slides and the story still read every number from a table, never from a notebook printout.

| # | Writes | Method | Answers | RQ |
|---|---|---|---|---|
| 1 | `fct_rq1_model_estimates` | Weighted linear probability model, year FE + state FE, HC1 and state-clustered SEs; robustness specs; logit AME check | "How big is the co-partisan premium once the national economy (year FE) is held fixed?" | RQ1 |
| 1 | `fct_rq1_year_effects` | Year fixed effects α<sub>t</sub> from the main spec | The common, non-partisan year component (feeds RQ2) | RQ1 → RQ2 |
| 2 | `fct_swing_timing_bootstrap` | Parametric noise bootstrap of the UMich monthly readings | Intervals for "share of the swing done at the election / inauguration" | RQ1 timing |
| 3 | `fct_pocketbook_gap_bootstrap` | Respondent bootstrap within ANES year | Intervals for the national gap, personal gap, difference and ratio | RQ3 |
| 3 | `fct_pocketbook_gap_model` | Stacked LPM (item × out-party) with income, employment, homeownership and demographic controls, respondent-clustered SEs | "Does the wallet-vs-economy difference survive controls?" | RQ3 |
| 4 | `fct_rq2_correlations` | Correlations of year-over-year changes, bootstrap over years | "Does the non-partisan component track fundamentals?" | RQ2 |
| 5 | `fct_say_vs_do_correlations` | Monthly correlations, moving-block bootstrap | "Does party sentiment move with spending?" | RQ4 |

## Pre-specification
Every spec, window and contrast below is listed in a constant at the top of its section **before** any estimate is printed, and every
one is written to its table whatever it shows. That is the plan's answer to "how do you know you didn't p-hack?". Nothing here is
chosen by looking at results.

## Decisions made in this notebook
1. **Pooled CES regressions use `weight_cumulative`**, the CES weight rescaled so each year counts equally (staging note: "only for
   pooled multi-year estimates"). With the year-specific `weight`, the 60k-respondent even years would outweigh the 17k odd years,
   and the odd years are exactly the post-inauguration contrasts. Year-specific weights and unweighted fits are robustness specs.
2. **Missing controls get their own "Missing" category instead of dropping the respondent.** The main spec and the no-controls spec
   then use exactly the same respondents, so "with and without controls" compares models, not samples. (CES coverage is high;
   ANES home ownership is 49% in 2020, where dropping would halve the sample.)
3. **Coefficients on binary outcomes are reported in percentage points** (× 100), matching how the story reports shares.
4. **Inference.** CES has no design variables: HC1 robust SEs with weights, and state-clustered SEs as the conservative bound (plan).
   Reported confidence intervals use the clustered SEs.
5. **Seeds are fixed** (`SEED`), so every bootstrap reproduces exactly on re-run.

## Run order
Notebooks 01 → 02 → 03 → this one. Each table is overwritten on re-run. Runtime on serverless: a few minutes, mostly the logit check.

## 0. Setup
`statsmodels` ships with most Databricks environments; the `%pip` cell makes sure. If it installs anything, the next cell restarts
Python so the new version is picked up.

In [0]:
%python
%pip install statsmodels

In [0]:
%python
%restart_python

In [0]:
import numpy as np
import pandas as pd
import patsy
import statsmodels.api as sm
from pyspark.sql import functions as F

# Same catalog/schema as notebooks 01-03. (Re-run this cell if a serverless session times out: TABLE_OR_VIEW_NOT_FOUND.)
try:
    spark.sql("USE CATALOG workspace")
    spark.sql("USE SCHEMA public_opinion_economy_case_study")
    print("Using workspace.public_opinion_economy_case_study")
except Exception as e:  # only expected outside Databricks (the local test harness has no Unity Catalog)
    print(f"Unity Catalog not available ({type(e).__name__}); using the session's current database")

SEED = 20261022          # fixed seed: every bootstrap reproduces exactly
N_BOOT = 2000            # bootstrap draws for respondent-level and year-level resampling
N_BOOT_NOISE = 5000      # draws for the (cheap, vectorized) UMich noise bootstrap


def write_table(pdf: pd.DataFrame, name: str, comment: str) -> None:
    """Write a pandas result to the schema, overwriting. NaN becomes SQL NULL so `IS NULL` works downstream."""
    sdf = spark.createDataFrame(pdf)
    for field in sdf.schema.fields:
        if field.dataType.typeName() in ("double", "float"):
            sdf = sdf.withColumn(field.name, F.when(F.isnan(F.col(field.name)), None).otherwise(F.col(field.name)))
    sdf.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(name)
    safe = comment.replace("'", "")
    try:  # table comment shows in Catalog Explorer (Unity Catalog syntax)
        spark.sql(f"COMMENT ON TABLE {name} IS '{safe}'")
    except Exception:
        try:  # fallback for plain Spark (local test harness)
            spark.sql(f"ALTER TABLE {name} SET TBLPROPERTIES ('comment' = '{safe}')")
        except Exception:
            pass                                 # a missing comment never blocks the pipeline
    print(f"wrote {name}: {len(pdf):,} rows")


def categorical_with_missing(s: pd.Series) -> pd.Series:
    """Decision 2: a missing control becomes its own category, so no respondent is dropped for a missing control."""
    return s.astype("object").where(s.notna(), "Missing").astype(str)

## 1. RQ1 — the co-partisan premium, holding the national economy fixed
The plan's model, on CES respondents 2016–2025:

> Better<sub>it</sub> = α<sub>t</sub> + β<sub>1</sub> Rep<sub>i</sub> + β<sub>2</sub> Dem<sub>i</sub> + β<sub>3</sub> CoPartisan<sub>it</sub> + γ′X<sub>it</sub> + δ<sub>s</sub> + ε<sub>it</sub>

* **α<sub>t</sub> (year FE)** absorbs everything national in a year, including the economy itself. So **β<sub>3</sub>**, the
  co-partisan premium, is identified only from *who holds the White House* changing (2017, 2021, 2025).
* Rep and Dem are relative to independents. CoPartisan = 1 if the respondent's party holds the White House (from `int_respondents`).
* X = age group, education, race, sex, income tercile, employment, homeownership (`int_respondents` harmonized controls, decision 2).
  δ = state FE.
* "Not sure" party identifiers are excluded (CoPartisan is undefined for them).

### Pre-specified specs (all written to `fct_rq1_model_estimates`)
| spec | Changes vs main | Robustness battery item |
|---|---|---|
| `main` | — | headline |
| `split_by_party` | β<sub>3</sub> split into Republican and Democratic co-partisan terms | asymmetry test (plan) |
| `ns_as_same` | outcome counts "not sure" as "same" | "Not sure" excluded vs folded |
| `leaners_as_independents` | strict party coding | leaners with parties vs as independents |
| `no_controls` | no X, no state FE | with and without controls |
| `excluding_covid` | drops 2020 and 2021 | excluding COVID |
| `year_specific_weights` | `weight` instead of `weight_cumulative` | weighting choice |
| `unweighted` | no weights | weighted vs unweighted |
| `outcome_worse` | outcome = "worse" (expect a negative β<sub>3</sub>) | mirror outcome |
| `outcome_score5` | outcome = 5-point score (scale points, not pp) | binary vs 5-point mean |
| `logit_ame` | logit, average marginal effect of CoPartisan | LPM vs logit |

In [0]:
RQ1_CONTROLS = ["age_group", "educ3", "race4", "sex", "income_tercile", "employment5", "owns_home"]

# spec name -> settings. Written down before any model is fit.
RQ1_SPECS = {
    "main":                    dict(outcome="econ_better",            strict=False, controls=True,  weight="weight_cumulative", years=None,        split=False),
    "split_by_party":          dict(outcome="econ_better",            strict=False, controls=True,  weight="weight_cumulative", years=None,        split=True),
    "ns_as_same":              dict(outcome="econ_better_ns_as_same", strict=False, controls=True,  weight="weight_cumulative", years=None,        split=False),
    "leaners_as_independents": dict(outcome="econ_better",            strict=True,  controls=True,  weight="weight_cumulative", years=None,        split=False),
    "no_controls":             dict(outcome="econ_better",            strict=False, controls=False, weight="weight_cumulative", years=None,        split=False),
    "excluding_covid":         dict(outcome="econ_better",            strict=False, controls=True,  weight="weight_cumulative", years="no_covid",  split=False),
    "year_specific_weights":   dict(outcome="econ_better",            strict=False, controls=True,  weight="weight",            years=None,        split=False),
    "unweighted":              dict(outcome="econ_better",            strict=False, controls=True,  weight=None,                years=None,        split=False),
    "outcome_worse":           dict(outcome="econ_worse",             strict=False, controls=True,  weight="weight_cumulative", years=None,        split=False),
    "outcome_score5":          dict(outcome="econ_score5",            strict=False, controls=True,  weight="weight_cumulative", years=None,        split=False),
}

# Only the columns the models need, CES only (ANES has no post-inauguration waves, so it cannot identify beta-3).
ces_cols = ["survey_year", "weight", "weight_cumulative", "state_abbr",
            "party3", "party3_strict", "co_partisan", "co_partisan_strict",
            "econ_better", "econ_worse", "econ_better_ns_as_same", "econ_score5"] + RQ1_CONTROLS
ces = spark.table("int_respondents").where("source = 'CES'").select(*ces_cols).toPandas()
for c in RQ1_CONTROLS:
    ces[c] = categorical_with_missing(ces[c])
print(f"CES respondents loaded: {len(ces):,}")


def rq1_design(spec: dict) -> tuple[pd.DataFrame, str]:
    """Build the analysis frame and formula for one spec."""
    party_col = "party3_strict" if spec["strict"] else "party3"
    cp_col = "co_partisan_strict" if spec["strict"] else "co_partisan"
    d = ces[ces[cp_col].notna() & ces[spec["outcome"]].notna()].copy()       # drops "Not sure" party and missing outcome
    if spec["years"] == "no_covid":
        d = d[~d["survey_year"].isin([2020, 2021])]
    d["y"] = d[spec["outcome"]].astype(float)
    d["rep"] = (d[party_col] == "Republican").astype(float)
    d["dem"] = (d[party_col] == "Democrat").astype(float)
    d["co_partisan"] = d[cp_col].astype(float)
    d["co_partisan_rep"] = d["co_partisan"] * d["rep"]                         # Republicans while a Republican is president
    d["co_partisan_dem"] = d["co_partisan"] * d["dem"]
    cp_terms = "co_partisan_rep + co_partisan_dem" if spec["split"] else "co_partisan"
    formula = f"y ~ C(survey_year) + rep + dem + {cp_terms}"
    if spec["controls"]:
        formula += " + " + " + ".join(f"C({c})" for c in RQ1_CONTROLS) + " + C(state_abbr)"
    return d, formula


def fit_wls(d: pd.DataFrame, formula: str, weight_col, cluster_col: str):
    """One WLS fit, two covariance estimators (HC1 and cluster-robust). Returns params, both SE vectors, and the fit."""
    y, X = patsy.dmatrices(formula, d, return_type="dataframe")
    w = d.loc[y.index, weight_col].to_numpy() if weight_col else np.ones(len(y))
    res = sm.WLS(y, X, weights=w).fit()
    groups = pd.factorize(d.loc[y.index, cluster_col])[0]
    hc1 = res.get_robustcov_results(cov_type="HC1")
    clu = res.get_robustcov_results(cov_type="cluster", groups=groups)
    names = list(X.columns)
    return (pd.Series(res.params.to_numpy(), index=names),
            pd.Series(hc1.bse, index=names),
            pd.Series(clu.bse, index=names),
            res, int(groups.max() + 1), X, w, groups)


rq1_rows, year_effect_rows = [], []
TERMS_OF_INTEREST = ["co_partisan", "co_partisan_rep", "co_partisan_dem", "rep", "dem"]

for spec_name, spec in RQ1_SPECS.items():
    d, formula = rq1_design(spec)
    params, se_hc1, se_cl, res, n_clusters, X, w, groups = fit_wls(d, formula, spec["weight"], "state_abbr")
    scale = 1.0 if spec["outcome"] == "econ_score5" else 100.0              # decision 3: percentage points for binaries
    for term in TERMS_OF_INTEREST:
        if term in params.index:
            est, s1, s2 = params[term] * scale, se_hc1[term] * scale, se_cl[term] * scale
            rq1_rows.append(dict(
                spec=spec_name, term=term, outcome=spec["outcome"],
                unit="scale points (1-5)" if scale == 1.0 else "percentage points",
                estimate=est, se_hc1=s1, se_cluster_state=s2,
                ci_low_cluster=est - 1.96 * s2, ci_high_cluster=est + 1.96 * s2,
                z_cluster=est / s2, n_obs=int(res.nobs), n_clusters=n_clusters, r_squared=res.rsquared,
                weight_used=spec["weight"] or "none", includes_controls=spec["controls"],
                is_main_spec=(spec_name == "main"), method="WLS linear probability model"))
    # Year effects from the main spec: alpha_t relative to 2016 (the omitted year), in percentage points
    if spec_name == "main":
        year_effect_rows.append(dict(survey_year=2016, year_effect_pp=0.0, se_cluster_state=0.0, is_reference_year=True))
        for name in params.index:
            if name.startswith("C(survey_year)[T."):
                yr = int(name.split("[T.")[1].rstrip("]"))
                year_effect_rows.append(dict(survey_year=yr, year_effect_pp=params[name] * 100,
                                             se_cluster_state=se_cl[name] * 100, is_reference_year=False))
    print(f"{spec_name:25s} n={int(res.nobs):>7,}  " +
          "  ".join(f"{t}={params[t] * scale:6.2f} ({se_cl[t] * scale:.2f})" for t in TERMS_OF_INTEREST if t in params.index))

**Logit check (plan: "Logit average marginal effects as a check").** Same main-spec design, binomial GLM with the pooled weights
as variance weights and state-clustered SEs. The AME of CoPartisan is β × the weighted mean of p(1 − p) (derivative form).
Its SE scales the coefficient's clustered SE by the same factor, which treats that factor as fixed: approximate, and labeled so.
If the LPM and the logit AME agree to within a few points, the linear model's simplicity costs nothing.
(statsmodels warns that clustered covariance is "not fully supported with var_weights": one more reason the AME's SE is labeled
approximate. The point estimate is the check; the LPM carries the inference.)

In [0]:
d, formula = rq1_design(RQ1_SPECS["main"])
y, X = patsy.dmatrices(formula, d, return_type="dataframe")
w = d.loc[y.index, "weight_cumulative"].to_numpy()
w = w / w.mean()                                                                # normalize so the GLM's scale is in respondents
groups = pd.factorize(d.loc[y.index, "state_abbr"])[0]
logit = sm.GLM(y, X, family=sm.families.Binomial(), var_weights=w).fit(cov_type="cluster", cov_kwds={"groups": groups})
p_hat = logit.predict(X)
dpdx_factor = np.average(p_hat * (1 - p_hat), weights=w)
ame = logit.params["co_partisan"] * dpdx_factor * 100
ame_se = logit.bse["co_partisan"] * dpdx_factor * 100
rq1_rows.append(dict(
    spec="logit_ame", term="co_partisan", outcome="econ_better", unit="percentage points",
    estimate=ame, se_hc1=np.nan, se_cluster_state=ame_se,
    ci_low_cluster=ame - 1.96 * ame_se, ci_high_cluster=ame + 1.96 * ame_se, z_cluster=ame / ame_se,
    n_obs=int(logit.nobs), n_clusters=int(groups.max() + 1), r_squared=np.nan,
    weight_used="weight_cumulative", includes_controls=True, is_main_spec=False,
    method="Logit average marginal effect (approximate SE)"))
print(f"logit AME of co_partisan = {ame:.2f} pp (approx. clustered SE {ame_se:.2f})")

rq1_estimates = pd.DataFrame(rq1_rows)
write_table(rq1_estimates, "fct_rq1_model_estimates",
            "RQ1 pooled CES model: co-partisan premium and party terms by pre-specified spec, percentage points, HC1 and state-clustered SEs, logit AME check.")

year_effects = pd.DataFrame(year_effect_rows).sort_values("survey_year")
year_effects["ci_low_cluster"] = year_effects["year_effect_pp"] - 1.96 * year_effects["se_cluster_state"]
year_effects["ci_high_cluster"] = year_effects["year_effect_pp"] + 1.96 * year_effects["se_cluster_state"]
year_effects["year_effect_chg_vs_prev_year"] = year_effects["year_effect_pp"].diff()
write_table(year_effects, "fct_rq1_year_effects",
            "RQ1 main-spec year fixed effects (percentage points vs 2016): the common, non-partisan component of CES economic ratings by year.")

## 2. RQ1 timing — bootstrap intervals for the share of the swing done at each date
`fct_swing_timing` has the point estimates. UMich publishes no standard errors, so the interval comes from the noise model of
notebook 03 (decision 6 there): every monthly reading = true value + noise with SD `noise_sd_level` for that series.

**Parametric bootstrap:** draw fresh noise for every month −4..+8 of every series, independently, `N_BOOT_NOISE` times; recompute
the baseline, the total swing and both shares exactly as the mart does (gap series from the perturbed party series). Report:
* 95% percentile intervals for `pct_swing_done_at_election` and `pct_swing_done_at_takeover`, **only when at least 95% of draws pass the
  mart's interpretability rule** (swing beyond noise, no reversal); otherwise NULL, with `pct_draws_interpretable` saying why;
* two probabilities that map onto the story's claims: the move had **started** by the election month (share > 0) and it **kept going**
  after the inauguration (share at takeover < 100%);
* a **stress test** with noise doubled (`*_2x_noise`), since the noise model is itself an estimate.

**Scope:** this captures measurement noise in the monthly readings. It does not treat the two transitions as a sample of possible
transitions; with n = 2, the write-up says "in both transitions", not "on average".

In [0]:
es = (spark.table("fct_event_study")
      .where("series IN ('Republican', 'Democrat', 'Independent', 'All respondents') "
             "AND months_from_election BETWEEN -4 AND 8 AND baseline_index IS NOT NULL")
      .select("event_key", "event_label", "event_type", "winning_party", "series", "party_role",
              "months_from_election", "sentiment_index", "noise_sd_level")
      .toPandas())
swing_points = spark.table("fct_swing_timing").toPandas().set_index(["event_key", "series"])

MONTHS = list(range(-4, 9))                      # -4..+8
IDX = {m: i for i, m in enumerate(MONTHS)}
BASE = [IDX[-4], IDX[-3], IDX[-2]]
rng = np.random.default_rng(SEED)


def shares_from_draws(sim: np.ndarray, band: float):
    """sim: draws x months. Returns share at election, share at takeover, interpretable flag (all per draw)."""
    change = sim - sim[:, BASE].mean(axis=1, keepdims=True)
    total = change[:, IDX[8]]
    with np.errstate(divide="ignore", invalid="ignore"):
        share_e = 100 * change[:, IDX[0]] / total
        share_t = 100 * change[:, IDX[2]] / total
    window = change[:, IDX[0]:IDX[8] + 1]
    reversed_ = ((np.sign(window) == -np.sign(total)[:, None]) & (np.abs(window) > band)).any(axis=1)
    interpretable = (np.abs(total) > band) & ~reversed_
    return share_e, share_t, interpretable


def summarize(share_e, share_t, ok):
    frac_ok = ok.mean()
    if frac_ok >= 0.95:
        e, t = share_e[ok], share_t[ok]
        return dict(pct_draws_interpretable=100 * frac_ok,
                    election_ci_low=np.percentile(e, 2.5), election_ci_high=np.percentile(e, 97.5),
                    takeover_ci_low=np.percentile(t, 2.5), takeover_ci_high=np.percentile(t, 97.5),
                    prob_move_started_by_election=np.mean(e > 0),
                    prob_swing_continued_after_takeover=np.mean(t < 100))
    return dict(pct_draws_interpretable=100 * frac_ok, election_ci_low=np.nan, election_ci_high=np.nan,
                takeover_ci_low=np.nan, takeover_ci_high=np.nan,
                prob_move_started_by_election=np.nan, prob_swing_continued_after_takeover=np.nan)


boot_rows = []
for event_key, ev in es.groupby("event_key"):
    # levels[series] = vector over MONTHS; sigma[series]
    levels, sigma = {}, {}
    for series, g in ev.groupby("series"):
        g = g.set_index("months_from_election").reindex(MONTHS)
        if g["sentiment_index"].isna().any():
            continue                                               # incomplete window: no bootstrap for this series
        levels[series] = g["sentiment_index"].to_numpy(float)
        sigma[series] = float(g["noise_sd_level"].dropna().iloc[0])
    winner = ev["winning_party"].iloc[0]
    loser = "Democrat" if winner == "Republican" else "Republican"
    for noise_mult, suffix in [(1.0, ""), (2.0, "_2x_noise")]:
        sims = {s: levels[s] + rng.normal(0, noise_mult * sigma[s], size=(N_BOOT_NOISE, len(MONTHS))) for s in levels}
        targets = {s: (sims[s], noise_mult * sigma[s]) for s in sims}
        if winner in sims and loser in sims:
            gap_sigma = noise_mult * np.sqrt(sigma[winner] ** 2 + sigma[loser] ** 2)
            targets["Winning minus losing party"] = (sims[winner] - sims[loser], gap_sigma)
        for series, (sim, sig) in targets.items():
            band = 2 * sig * np.sqrt(4 / 3)                         # same band as the mart
            stats = summarize(*shares_from_draws(sim, band))
            key = (event_key, series)
            row = next((r for r in boot_rows if (r["event_key"], r["series"]) == key), None)
            if row is None:
                pt = swing_points.loc[key] if key in swing_points.index else None
                row = dict(event_key=event_key, event_label=ev["event_label"].iloc[0], event_type=ev["event_type"].iloc[0],
                           series=series, noise_sd_level=sig / noise_mult, n_draws=N_BOOT_NOISE,
                           pct_swing_done_at_election=None if pt is None else pt["pct_swing_done_at_election"],
                           pct_swing_done_at_takeover=None if pt is None else pt["pct_swing_done_at_takeover"])
                boot_rows.append(row)
            row.update({f"{k}{suffix}": v for k, v in stats.items()})

swing_boot = pd.DataFrame(boot_rows)
swing_boot["method"] = "Parametric bootstrap of monthly measurement noise (robust MAD-based SD per series), percentile 95% CI"
for c in ["pct_swing_done_at_election", "pct_swing_done_at_takeover"]:
    swing_boot[c] = pd.to_numeric(swing_boot[c], errors="coerce")
write_table(swing_boot, "fct_swing_timing_bootstrap",
            "Bootstrap intervals for the share of each UMich series swing complete at the election month and the takeover month, with probabilities that the move began by the election and continued after the inauguration; 1x and 2x noise.")

## 3. RQ3 — uncertainty and controls for the wallet-vs-economy gap
### 3a. Respondent bootstrap (`fct_pocketbook_gap_bootstrap`)
Within each ANES year, resample respondents with replacement `N_BOOT` times and recompute the weighted national gap, personal gap,
their difference and both ratios, exactly as `fct_pocketbook_gap` does. Percentile 95% intervals.
**Caveat:** the cumulative file has no strata or PSUs (staging gotcha 1), so this treats ANES as a simple random sample and
understates uncertainty somewhat. Merging the design variables from the yearly study files is a listed data gap.

In [0]:
anes_cols = ["survey_year", "respondent_key", "weight", "partisan_role", "partisan_role_strict",
             "econ_worse", "personal_worse", "econ_worse_ns_as_same", "personal_worse_ns_as_same"] + RQ1_CONTROLS
anes = spark.table("int_respondents").where("source = 'ANES'").select(*anes_cols).toPandas()

RQ3_SPECS = {  # (outcome spec label, national column, personal column)
    "Main: do not know excluded":      ("econ_worse",            "personal_worse"),
    "Robustness: do not know in base": ("econ_worse_ns_as_same", "personal_worse_ns_as_same"),
}
RQ3_PARTY_DEFS = {"Leaners with parties": "partisan_role", "Leaners as independents": "partisan_role_strict"}


def weighted_gap(out_flag, values, weights):
    """Out-party minus in-party weighted % worse, ignoring NaN values (each item's own base)."""
    valid = ~np.isnan(values)
    o, i = valid & out_flag, valid & ~out_flag
    return 100 * (np.sum(values[o] * weights[o]) / np.sum(weights[o]) - np.sum(values[i] * weights[i]) / np.sum(weights[i]))


rng = np.random.default_rng(SEED)
pb_rows = []
for (spec_label, (nat_col, per_col)), (def_label, role_col) in [(s, p) for s in RQ3_SPECS.items() for p in RQ3_PARTY_DEFS.items()]:
    for year, g in anes[anes[role_col].isin(["In-party", "Out-party"])].groupby("survey_year"):
        out = (g[role_col] == "Out-party").to_numpy()
        nat = g[nat_col].astype(float).to_numpy()
        per = g[per_col].astype(float).to_numpy()
        wts = g["weight"].astype(float).to_numpy()
        n = len(g)
        draws = np.empty((N_BOOT, 2))
        for b in range(N_BOOT):
            ix = rng.integers(0, n, n)
            draws[b] = (weighted_gap(out[ix], nat[ix], wts[ix]), weighted_gap(out[ix], per[ix], wts[ix]))
        nat_gap, per_gap = weighted_gap(out, nat, wts), weighted_gap(out, per, wts)
        diff, ratio_np, ratio_pn = draws[:, 0] - draws[:, 1], draws[:, 0] / draws[:, 1], draws[:, 1] / draws[:, 0]
        pb_rows.append(dict(
            survey_year=int(year), outcome_spec=spec_label, party_definition=def_label,
            is_main_spec=(spec_label.startswith("Main") and def_label == "Leaners with parties"),
            national_gap_pts=nat_gap, national_gap_ci_low=np.percentile(draws[:, 0], 2.5), national_gap_ci_high=np.percentile(draws[:, 0], 97.5),
            personal_gap_pts=per_gap, personal_gap_ci_low=np.percentile(draws[:, 1], 2.5), personal_gap_ci_high=np.percentile(draws[:, 1], 97.5),
            gap_difference_pts=nat_gap - per_gap, gap_difference_ci_low=np.percentile(diff, 2.5), gap_difference_ci_high=np.percentile(diff, 97.5),
            prob_national_gap_larger=np.mean(diff > 0),
            national_to_personal_ratio=nat_gap / per_gap,
            national_to_personal_ratio_ci_low=np.percentile(ratio_np, 2.5), national_to_personal_ratio_ci_high=np.percentile(ratio_np, 97.5),
            personal_to_national_ratio=per_gap / nat_gap,
            personal_to_national_ratio_ci_low=np.percentile(ratio_pn, 2.5), personal_to_national_ratio_ci_high=np.percentile(ratio_pn, 97.5),
            n_respondents=n, n_draws=N_BOOT,
            method="Respondent bootstrap within year (no strata/PSU in cumulative file), percentile 95% CI"))

pocketbook_boot = pd.DataFrame(pb_rows)
write_table(pocketbook_boot, "fct_pocketbook_gap_bootstrap",
            "RQ3 bootstrap: ANES national and own-finances partisan gaps, their difference and ratios, with respondent-bootstrap 95% intervals, by year, outcome spec and party definition.")

### 3b. Controlled model (`fct_pocketbook_gap_model`)
The plan's objection to answer: *maybe out-partisans really were worse off* (an income realignment between the parties could create a
genuine pocketbook gap). So, per ANES year, each respondent contributes **two rows** (national item, own-finances item):

> Worse<sub>ij</sub> = a + b<sub>1</sub> OutParty<sub>i</sub> + b<sub>2</sub> National<sub>j</sub> + b<sub>3</sub> (OutParty × National)<sub>ij</sub> + γ′X<sub>i</sub> + e<sub>ij</sub>

* **b<sub>1</sub>** = the own-finances partisan gap net of controls; **b<sub>1</sub> + b<sub>3</sub>** = the national gap net of
  controls; **b<sub>3</sub>** = how much more partisan "the economy" is than the wallet (RQ3's comparison). RQ3 holds if b<sub>3</sub> > 0.
* X = income tercile, employment, homeownership (the plan's named controls) plus age group, education, race, sex; missing → own
  category (decision 2). Controls enter once per respondent, so they shift both items equally.
* SEs clustered by respondent (two rows each). Weighted; main outcome spec; leaners with parties.
* **Not yet controlled:** recent layoff (`VCF0156`). It is in `stg_anes` (`laid_off_last_6mo`) but was not carried into
  `int_respondents`; adding it there is a one-line change if the panel asks.

In [0]:
ctrl_rows = []
for year, g in anes[anes["partisan_role"].isin(["In-party", "Out-party"])].groupby("survey_year"):
    base = g.copy()
    for c in RQ1_CONTROLS:
        base[c] = categorical_with_missing(base[c])
    base["out_party"] = (base["partisan_role"] == "Out-party").astype(float)
    long = pd.concat([
        base.assign(national=1.0, worse=base["econ_worse"]),
        base.assign(national=0.0, worse=base["personal_worse"]),
    ], ignore_index=True)                     # fresh row labels: each respondent now has two distinct rows
    long = long[long["worse"].notna()].copy()
    long["worse"] = long["worse"].astype(float)
    formula = "worse ~ out_party * national + " + " + ".join(f"C({c})" for c in RQ1_CONTROLS)
    for with_controls in (False, True):
        f = formula if with_controls else "worse ~ out_party * national"
        yv, Xv = patsy.dmatrices(f, long, return_type="dataframe")
        res = sm.WLS(yv, Xv, weights=long.loc[yv.index, "weight"].astype(float).to_numpy()).fit()
        clu = res.get_robustcov_results(cov_type="cluster", groups=pd.factorize(long.loc[yv.index, "respondent_key"])[0])
        params, bse = pd.Series(res.params.to_numpy(), index=Xv.columns), pd.Series(clu.bse, index=Xv.columns)
        cov = pd.DataFrame(clu.cov_params(), index=Xv.columns, columns=Xv.columns)
        nat_gap = params["out_party"] + params["out_party:national"]
        nat_se = np.sqrt(cov.loc["out_party", "out_party"] + cov.loc["out_party:national", "out_party:national"]
                         + 2 * cov.loc["out_party", "out_party:national"])
        for term, est, se in [("personal_gap (b1)", params["out_party"], bse["out_party"]),
                              ("national_gap (b1 + b3)", nat_gap, nat_se),
                              ("national_minus_personal (b3)", params["out_party:national"], bse["out_party:national"])]:
            ctrl_rows.append(dict(survey_year=int(year), term=term, with_controls=with_controls,
                                  estimate_pp=100 * est, se_cluster_respondent=100 * se,
                                  ci_low=100 * (est - 1.96 * se), ci_high=100 * (est + 1.96 * se),
                                  n_rows=int(res.nobs), n_respondents=int(long.loc[yv.index, "respondent_key"].nunique()),
                                  outcome_spec="Main: do not know excluded", party_definition="Leaners with parties"))

pocketbook_model = pd.DataFrame(ctrl_rows)
write_table(pocketbook_model, "fct_pocketbook_gap_model",
            "RQ3 controlled model: ANES own-finances gap, national gap and their difference (pp) from a stacked item x out-party LPM, with and without income, employment, homeownership and demographic controls; respondent-clustered SEs.")

## 4. RQ2 — does the non-partisan component track the fundamentals? (`fct_rq2_correlations`)
Two "non-partisan" estimates (plan): **independents' net rating** (A2) and the **year effects** from section 1. Each is correlated,
in **year-over-year changes** (plan: levels correlations flip sign between terms), with changes in the fundamentals over the trailing
window before fieldwork (`fct_rq2_annual_panel`).

* **Pre-specified grid:** 3 perception measures × 7 fundamentals × 3 windows; the 3-month window is primary (A5). All are written.
* **Intervals:** bootstrap over years (resample the year-pairs), percentile 95%. With 6–9 points, intervals will be wide; the plan
  labels this descriptive, and so does the table (`is_descriptive_only`).
* **The salience contrast (one, pre-specified):** r(gasoline) − r(core) for independents' net rating, 3-month window, with its own
  bootstrap interval. CPI component changes start with the 2018 wave (component y/y starts 2017), so this uses fewer years.
* Expected signs if perceptions track fundamentals: negative for inflation, unemployment and prices; positive for real earnings.
* **Multiple comparisons.** The grid holds 63 correlations; with 95% intervals, about three would exclude zero by chance alone. Only the
  salience contrast is confirmatory. Any other interval that excludes zero is a lead for a follow-up, labeled exploratory (plan:
  "everything else labeled exploratory, with Holm adjustment where tests are reported").

In [0]:
RQ2_PERCEPTIONS = {"ind_net_rating_chg": "Independents' net rating", "ind_pct_better_chg": "Independents' % better",
                   "year_effect_chg_vs_prev_year": "Year effect (RQ1 main spec)"}
RQ2_FUNDAMENTALS = {"inflation_chg_vs_prev_wave": "Headline inflation", "unemployment_chg_vs_prev_wave": "Unemployment",
                    "gasoline_yoy_chg_vs_prev_wave": "Gasoline CPI", "food_at_home_yoy_chg_vs_prev_wave": "Food-at-home CPI",
                    "core_yoy_chg_vs_prev_wave": "Core CPI", "shelter_yoy_chg_vs_prev_wave": "Shelter CPI",
                    "real_earnings_yoy_chg_vs_prev_wave": "Real hourly earnings"}

panel = spark.table("fct_rq2_annual_panel").toPandas().merge(
    year_effects[["survey_year", "year_effect_chg_vs_prev_year"]], on="survey_year", how="left")

rng = np.random.default_rng(SEED)


def boot_corr(x: np.ndarray, y: np.ndarray, n_boot: int = N_BOOT):
    """Pearson r and percentile CI from resampling (x, y) pairs. Draws with < 3 distinct points or zero variance are skipped."""
    n = len(x)
    rs = []
    for _ in range(n_boot):
        ix = rng.integers(0, n, n)
        if len(np.unique(ix)) < 3 or np.std(x[ix]) == 0 or np.std(y[ix]) == 0:
            continue
        rs.append(np.corrcoef(x[ix], y[ix])[0, 1])
    return np.array(rs)


rq2_rows = []
for window, gw in panel.groupby("window_months"):
    for pcol, plabel in RQ2_PERCEPTIONS.items():
        for fcol, flabel in RQ2_FUNDAMENTALS.items():
            sub = gw[[pcol, fcol, "survey_year"]].dropna()
            x, yv = sub[fcol].to_numpy(float), sub[pcol].to_numpy(float)
            if len(sub) < 4:
                continue
            r = np.corrcoef(x, yv)[0, 1]
            rs = boot_corr(x, yv)
            rq2_rows.append(dict(window_months=int(window), is_primary_window=bool(window == 3),
                                 perception_measure=plabel, fundamental=flabel, r=r,
                                 ci_low=np.percentile(rs, 2.5), ci_high=np.percentile(rs, 97.5),
                                 n_year_pairs=len(sub), first_wave=int(sub["survey_year"].min()), last_wave=int(sub["survey_year"].max()),
                                 is_salience_contrast_input=(fcol in ("gasoline_yoy_chg_vs_prev_wave", "core_yoy_chg_vs_prev_wave")),
                                 is_descriptive_only=True))

# The one pre-specified salience contrast: r(gas) - r(core), independents' net rating, 3-month window, same years for both.
sub = panel[panel["window_months"] == 3][["ind_net_rating_chg", "gasoline_yoy_chg_vs_prev_wave", "core_yoy_chg_vs_prev_wave"]].dropna()
yv, g_, c_ = (sub[c].to_numpy(float) for c in sub.columns)
diffs = []
for _ in range(N_BOOT):
    ix = rng.integers(0, len(sub), len(sub))
    if len(np.unique(ix)) < 3 or min(np.std(yv[ix]), np.std(g_[ix]), np.std(c_[ix])) == 0:
        continue
    diffs.append(np.corrcoef(g_[ix], yv[ix])[0, 1] - np.corrcoef(c_[ix], yv[ix])[0, 1])
rq2_rows.append(dict(window_months=3, is_primary_window=True, perception_measure="Independents' net rating",
                     fundamental="SALIENCE CONTRAST: r(gasoline) - r(core)",
                     r=np.corrcoef(g_, yv)[0, 1] - np.corrcoef(c_, yv)[0, 1],
                     ci_low=np.percentile(diffs, 2.5), ci_high=np.percentile(diffs, 97.5), n_year_pairs=len(sub),
                     first_wave=np.nan, last_wave=np.nan, is_salience_contrast_input=False, is_descriptive_only=True))

rq2 = pd.DataFrame(rq2_rows)
rq2["method"] = "Pearson r of year-over-year changes; bootstrap over years, percentile 95% CI; descriptive (n <= 9)"
write_table(rq2, "fct_rq2_correlations",
            "RQ2: correlations between year-over-year changes in non-partisan perception measures (independents, RQ1 year effects) and changes in fundamentals, by window, with bootstrap intervals and the gas-vs-core salience contrast.")

## 5. RQ4 — does party sentiment move with spending? (`fct_say_vs_do_correlations`)
Contemporaneous correlations between month-over-month changes in each sentiment series and real PCE growth (and the change in the
saving rate), from `fct_say_vs_do_monthly`. Monthly series are autocorrelated, so the interval is a **moving-block bootstrap**
(blocks of 6 months). Lead-lag correlations are deliberately not computed: the plan defers them until the lags are pre-specified.
Aggregate data cannot show whether partisans spend differently (that is the Imprint question); this is context.

**Two samples:** all months, and the plan's pre-specified COVID robustness (excluding 2020–2021). The full-sample index's correlation
with spending is driven by March–April 2020, when sentiment and spending collapsed together; the party series, which moved on a
different timetable that spring, show essentially none in either sample.

In [0]:
say_do = spark.table("fct_say_vs_do_monthly").orderBy("month").toPandas()
SAY = {"sentiment_rep_mom_chg": "Republican sentiment", "sentiment_dem_mom_chg": "Democratic sentiment",
       "sentiment_ind_mom_chg": "Independent sentiment", "ics_all_mom_chg": "All respondents' sentiment"}
DO = {"pce_real_mom_pct": "Real PCE growth (% m/m)", "personal_saving_rate_mom_chg": "Saving rate change (pts m/m)"}
BLOCK = 6
rng = np.random.default_rng(SEED)


def block_boot_corr(x: np.ndarray, y: np.ndarray, block: int = BLOCK, n_boot: int = N_BOOT):
    n = len(x)
    starts_max = n - block
    rs = []
    for _ in range(n_boot):
        starts = rng.integers(0, starts_max + 1, int(np.ceil(n / block)))
        ix = np.concatenate([np.arange(s, s + block) for s in starts])[:n]
        rs.append(np.corrcoef(x[ix], y[ix])[0, 1])
    return np.array(rs)


# Two samples: all months, and the plan's pre-specified COVID robustness ("excluding 2020-2021").
# Why it matters here: the all-respondents index fell 12 and 17 points in March-April 2020 as real PCE fell 7% and 11%;
# those two months alone carry most of that series' correlation (0.42 with them, about 0.18 without March-June 2020).
SAMPLES = {"All months": lambda d: d,
           "Excluding 2020-2021 (plan robustness)": lambda d: d[~pd.to_datetime(d["month"]).dt.year.isin([2020, 2021])]}

sd_rows = []
for sample_label, sample_filter in SAMPLES.items():
    for scol, slabel in SAY.items():
        for dcol, dlabel in DO.items():
            sub = sample_filter(say_do[["month", scol, dcol]].dropna())
            x, yv = sub[scol].to_numpy(float), sub[dcol].to_numpy(float)
            rs = block_boot_corr(x, yv)      # blocks can straddle the removed years; acceptable for a robustness view
            sd_rows.append(dict(sample=sample_label, say_measure=slabel, do_measure=dlabel, r=np.corrcoef(x, yv)[0, 1],
                                ci_low=np.percentile(rs, 2.5), ci_high=np.percentile(rs, 97.5), n_months=len(sub),
                                first_month=sub["month"].min(), last_month=sub["month"].max(),
                                method=f"Pearson r of monthly changes; moving-block bootstrap (block = {BLOCK} months), percentile 95% CI"))

say_do_corr = pd.DataFrame(sd_rows)
write_table(say_do_corr, "fct_say_vs_do_correlations",
            "RQ4 aggregate: correlation between month-over-month changes in UMich sentiment by party and real PCE growth or saving-rate changes, with moving-block bootstrap intervals.")

## 6. QA and headline read-out
Prints the numbers a reviewer would check first. Expected patterns (not targets): the co-partisan premium is large and positive in
every spec (negative for `outcome_worse`); the split-by-party terms are similar in size (symmetry); the logit AME is close to the
LPM; the swing-timing intervals for the parties exclude 0% at the takeover and exclude 100% at the takeover; the RQ3 difference
interval excludes 0 in every year; RQ2 and RQ4 intervals are wide.

In [0]:
with pd.option_context("display.width", 250, "display.max_columns", 30, "display.float_format", "{:.2f}".format):
    print("\n== RQ1: co-partisan premium by spec (pp; clustered 95% CI) ==")
    print(rq1_estimates[rq1_estimates["term"].str.startswith("co_partisan")]
          [["spec", "term", "unit", "estimate", "se_hc1", "se_cluster_state", "ci_low_cluster", "ci_high_cluster", "n_obs"]].to_string(index=False))
    print("\n== RQ1: year effects (pp vs 2016) ==")
    print(year_effects.to_string(index=False))
    print("\n== Swing timing bootstrap (presidential, 1x noise) ==")
    print(swing_boot[swing_boot["event_type"] == "Presidential election"]
          [["event_label", "series", "pct_swing_done_at_election", "election_ci_low", "election_ci_high",
            "pct_swing_done_at_takeover", "takeover_ci_low", "takeover_ci_high", "pct_draws_interpretable",
            "prob_move_started_by_election", "prob_swing_continued_after_takeover", "takeover_ci_low_2x_noise", "takeover_ci_high_2x_noise"]]
          .to_string(index=False))
    print("\n== RQ3 bootstrap (main spec) ==")
    print(pocketbook_boot[pocketbook_boot["is_main_spec"]]
          [["survey_year", "national_gap_pts", "national_gap_ci_low", "national_gap_ci_high", "personal_gap_pts",
            "personal_gap_ci_low", "personal_gap_ci_high", "gap_difference_ci_low", "gap_difference_ci_high",
            "national_to_personal_ratio", "national_to_personal_ratio_ci_low", "national_to_personal_ratio_ci_high"]].to_string(index=False))
    print("\n== RQ3 controlled model ==")
    print(pocketbook_model.to_string(index=False))
    print("\n== RQ2 primary window (3 months) ==")
    print(rq2[rq2["is_primary_window"]][["perception_measure", "fundamental", "r", "ci_low", "ci_high", "n_year_pairs"]].to_string(index=False))
    print("\n== RQ4 say vs do ==")
    print(say_do_corr[["sample", "say_measure", "do_measure", "r", "ci_low", "ci_high", "n_months"]].to_string(index=False))